In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay
)
import seaborn as sns
import matplotlib.pyplot as plt

In [21]:
df = pd.read_csv('Group Work_Pipeline_Last.csv')

In [22]:
X = df.drop(columns=["is_fit"])
y = df["is_fit"]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [32]:
model = GaussianNB()
param_grid = {"var_smoothing": [1e-9, 1e-8, 1e-7]}

In [33]:
grid = GridSearchCV(model, param_grid, cv=5, scoring="accuracy", return_train_score=True)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=GaussianNB(),
             param_grid={'var_smoothing': [1e-09, 1e-08, 1e-07]},
             return_train_score=True, scoring='accuracy')

In [34]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_pred)

print("\n=== Test Set Results ===")
print(f"Accuracy: {test_acc:.3f}")
print(f"Precision: {test_precision:.3f}")
print(f"Recall: {test_recall:.3f}")
print(f"F1 Score: {test_f1:.3f}")
print(f"ROC-AUC: {test_auc:.3f}")


=== Test Set Results ===
Accuracy: 0.790
Precision: 0.789
Recall: 0.647
F1 Score: 0.711
ROC-AUC: 0.766


In [35]:
cv_f1 = cross_val_score(best_model, X_train, y_train, cv=5, scoring="f1")
print("\nMean Cross-Validation F1 Score:", round(cv_f1.mean(), 3))


Mean Cross-Validation F1 Score: 0.721


In [36]:
results = pd.DataFrame(grid.cv_results_)[
    ["params", "mean_test_score", "rank_test_score"]
].sort_values(by="rank_test_score")

print("\n=== Tuned Model Comparison ===")
print(results)

# Optional: pretty table
results["mean_test_score"] = (results["mean_test_score"] * 100).round(2)
print("\nAccuracy of Each Tuned Version:")
print(results.rename(columns={"mean_test_score": "CV_Accuracy_%"}))


=== Tuned Model Comparison ===
                     params  mean_test_score  rank_test_score
0  {'var_smoothing': 1e-09}         0.796414                1
1  {'var_smoothing': 1e-08}         0.796414                1
2  {'var_smoothing': 1e-07}         0.796414                1

Accuracy of Each Tuned Version:
                     params  CV_Accuracy_%  rank_test_score
0  {'var_smoothing': 1e-09}          79.64                1
1  {'var_smoothing': 1e-08}          79.64                1
2  {'var_smoothing': 1e-07}          79.64                1


In [37]:
print("Best Parameters:", grid.best_params_)
print("Mean Cross-Validation Accuracy:", round(grid.best_score_ * 100, 2), "%")

Best Parameters: {'var_smoothing': 1e-09}
Mean Cross-Validation Accuracy: 79.64 %


In [ ]:
##“I trained the Naïve Bayes model three times by changing the parameter var_smoothing. All versions produced the same Cross-Validation Accuracy of 79.64%.This shows that the Naïve Bayes model is stable for this dataset — even when the parameter changes, the performance remains the same. Therefore, I selected var_smoothing = 1e-9 as the final best version.”